

Goal: \
#1 Calculate correspondance between targets and perceivers (correlations) for heart rate and heart rate variability

In [1]:
# libraries 

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy.stats import spearmanr
from scipy.stats import norm

In [2]:
# lists

# targets
targets = ['P03_joy', 'P29_joy', 'P34_joy', 'P42_joy', 'P44_joy',
           'P12_sadness', 'P15_sadness', 'P27_sadness', 'P34_sadness', 'P44_sadness', 
           'P03_anger', 'P15_anger', 'P22_anger', 'P38_anger', 'P42_anger',
           'P17_pride', 'P22_pride', 'P29_pride', 'P42_pride', 'P44_pride',
           'P03_shame', 'P22_shame', 'P37_shame', 'P38_shame', 'P45_shame', 
           'P12_neutral', 'P17_neutral', 'P29_neutral', 'P34_neutral', 'P38_neutral']

# perceivers
perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23",
              "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# data folder
data_folder = r"physiological_data/ECG_cleaned_data"

Check some stats about the number of ectopic beats in the distribution

In [3]:

ectopic_data = []

for target in targets:
    for perceiver in perceivers:
        
        filename = f"{perceiver}_{target}_RR.json"
        file_path = os.path.join(data_folder, filename)

        if not os.path.exists(file_path):
            continue  # perceiver didn't watch this target

        with open(file_path) as file:
            data = json.load(file)
            rr_intervals = np.array(data['RR_intervals'])
            ectopic = data.get('ectopic', 0) # default to 0
            num_beats = np.count_nonzero(~np.isnan(rr_intervals)) + ectopic # number of normal beats + ectopic beats (total)

        ectopic_data.append({
            "perceiver": perceiver,
            "target": target,
            "num_ectopic": ectopic,
            "num_beats": num_beats
        })

ectopic_data = pd.DataFrame(ectopic_data)

# group per perceiver
summary = (
    ectopic_data
    .groupby("perceiver")[["num_ectopic", "num_beats"]]
    .sum()
    .reset_index()
)
summary["percent_ectopic"] = (summary["num_ectopic"] / summary["num_beats"]) * 100

# print data
percent_ectopics = summary["percent_ectopic"].values
print("----- ECTOPIC STATS BY PERCEIVER -----")
print(f"Mean % ectopic: {np.nanmean(percent_ectopics):.2f}%")
print(f"Min-Max: {np.nanmin(percent_ectopics):.2f}% - {np.nanmax(percent_ectopics):.2f}%")
print("Perceivers with >5% ectopic:")
print(summary[summary["percent_ectopic"] > 5]["perceiver"].tolist())


##### now do the same without S01 and S18

cleaned_perceivers = ["S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S20", "S23", "S24", "S28", "S34", 
                      "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"] # removing S01 and S18

ectopic_data = []

for target in targets:
    for perceiver in cleaned_perceivers:
        
        filename = f"{perceiver}_{target}_RR.json"
        file_path = os.path.join(data_folder, filename)

        if not os.path.exists(file_path):
            continue  # perceiver didn't watch this target

        with open(file_path) as file:
            data = json.load(file)
            rr_intervals = np.array(data['RR_intervals'])
            ectopic = data.get('ectopic', 0) # default to 0
            num_beats = np.count_nonzero(~np.isnan(rr_intervals)) + ectopic # number of normal beats + ectopic beats (total)

        ectopic_data.append({
            "perceiver": perceiver,
            "target": target,
            "num_ectopic": ectopic,
            "num_beats": num_beats
        })

ectopic_data = pd.DataFrame(ectopic_data)

# group per perceiver
summary = (
    ectopic_data
    .groupby("perceiver")[["num_ectopic", "num_beats"]]
    .sum()
    .reset_index()
)
summary["percent_ectopic"] = (summary["num_ectopic"] / summary["num_beats"]) * 100

# print data
percent_ectopics = summary["percent_ectopic"].values
print("\n ----- ECTOPIC STATS BY PERCEIVER (removing S01 and S18) -----")
print(f"Mean % ectopic: {np.nanmean(percent_ectopics):.2f}%")
print(f"Min-Max: {np.nanmin(percent_ectopics):.2f}% - {np.nanmax(percent_ectopics):.2f}%")
print("Perceivers with >5% ectopic:")
print(summary[summary["percent_ectopic"] > 5]["perceiver"].tolist())


----- ECTOPIC STATS BY PERCEIVER -----
Mean % ectopic: 1.58%
Min-Max: 0.00% - 25.91%
Perceivers with >5% ectopic:
['S01', 'S18']

 ----- ECTOPIC STATS BY PERCEIVER (removing S01 and S18) -----
Mean % ectopic: 0.01%
Min-Max: 0.00% - 0.10%
Perceivers with >5% ectopic:
[]


Next, prepare an Excel sheet with different HRV metrics for each target-perceiver pair.

Calculation on cleaned data. Non-smoothed for now.

In [4]:
HRV_target_data = []
HRV_perceiver_data = []

for target in targets:
        
    with open(os.path.join(data_folder, f"{target}_RR.json")) as f:
        target_data = json.load(f)

    target_peak_times = np.array(target_data['peak_times'])
    target_time_diffs = np.array(target_data['RR_intervals'])
   
    target_mean_RR = np.nanmean(target_time_diffs) # mean RR interval (ignoring NaNs)
    target_sdnn = np.nanstd(target_time_diffs) # SD RR interval
    target_mean_HR = np.nanmean(60 / np.array(target_time_diffs)) # mean HR (bpm)
    target_sdHR = np.nanstd(60 / np.array(target_time_diffs)) # SD HR (bpm)
    target_minHR = np.nanmin(60 / np.array(target_time_diffs)) # min HR
    target_maxHR = np.nanmax(60 / np.array(target_time_diffs)) # max HR
    target_rmssd = np.sqrt(np.nanmean(np.square(np.diff(target_time_diffs)))) # RMSSD 
    target_nn50 = np.sum(np.abs(np.diff(target_time_diffs)) > .05)*1 # NN50
    target_pnn50 = 100 * np.sum((np.abs(np.diff(target_time_diffs)) > .05)*1) / len(target_time_diffs) # PNN50
    
    HRV_target_data.append({
            "target": target,
            "target_mean_RR": target_mean_RR,
            "target_sdnn": target_sdnn,
            "target_mean_HR": target_mean_HR,
            "target_sdHR": target_sdHR,
            "target_minHR": target_minHR,
            "target_maxHR": target_maxHR,
            "target_rmssd": target_rmssd,
            "target_nn50": target_nn50,
            "target_pnn50": target_pnn50})
       
    for perceiver in perceivers:
        
        perceiver_path = os.path.join(data_folder, f"{perceiver}_{target}_RR.json")
        try:
            with open(perceiver_path) as f:
                perceiver_data = json.load(f)
        except FileNotFoundError:
            continue
        
        perceiver_peak_times = np.array(perceiver_data['peak_times'])
        perceiver_time_diffs = np.array(perceiver_data['RR_intervals'])
        
        if len(perceiver_time_diffs) > 0: # if the evaluation exists
            
            perceiver_mean_RR = np.nanmean(perceiver_time_diffs) # mean RR interval
            perceiver_sdnn = np.nanstd(perceiver_time_diffs) # SD RR interval
            perceiver_mean_HR = np.nanmean(60 / np.array(perceiver_time_diffs)) # mean HR (bpm)
            perceiver_sdHR = np.nanstd(60 / np.array(perceiver_time_diffs)) # SD HR (bpm)
            perceiver_minHR = np.nanmin(60 / np.array(perceiver_time_diffs)) # min HR
            perceiver_maxHR = np.nanmax(60 / np.array(perceiver_time_diffs)) # max HR
            perceiver_rmssd = np.sqrt(np.nanmean(np.square(np.diff(perceiver_time_diffs)))) # RMSSD 
            perceiver_nn50 = np.sum(np.abs(np.diff(perceiver_time_diffs)) > .05)*1 # NN50
            perceiver_pnn50 = 100 * np.sum((np.abs(np.diff(perceiver_time_diffs)) > .05)*1) / len(perceiver_time_diffs) # PNN50

            HRV_perceiver_data.append({
                "perceiver": perceiver,
                "evalaution_of": target,
                "perceiver_mean_RR": perceiver_mean_RR,
                "perceiver_sdnn": perceiver_sdnn,
                "perceiver_mean_HR": perceiver_mean_HR,
                "perceiver_sdHR": perceiver_sdHR,
                "perceiver_minHR": perceiver_minHR,
                "perceiver_maxHR": perceiver_maxHR,
                "perceiver_rmssd": perceiver_rmssd,
                "perceiver_nn50": perceiver_nn50,
                "perceiver_pnn50": perceiver_pnn50})
                
pd.DataFrame(HRV_target_data).to_csv(r"outputs/HRV_metrics_target.csv", index=False)
pd.DataFrame(HRV_perceiver_data).to_csv(r"outputs/HRV_metrics_perceiver.csv", index=False)

Functions to make HR and HRV smoothed data. (bpm and rmssd)

In [ ]:
# compute RMSSD
def RMSSD(RR_intervals):
    return np.sqrt(np.nanmean(np.square(np.diff(RR_intervals))))

def sliding_HRV(time, RR_intervals, HALF_WINDOW = 10):
    
    RR_intervals = np.array(RR_intervals)
    time = np.array(time)
    
    valid = ~np.isnan(RR_intervals) # remove NaNs
    RR_intervals = RR_intervals[valid]
    time = time[valid]
    
    # create normalized time (start = 1, end = last timestamp) 
    max_second = int(time[-1]) # keep only the full second of the last value in time
    normalized_time = list(range(1, max_second + 1))  # 1, 2, 3, ..., max_second
    
    smoothed_rmssd = [] # store results
    
    for t in normalized_time: 
        
        in_window = (time >= (t - HALF_WINDOW)) & (time <= (t + HALF_WINDOW))
        local_rr_intervals = RR_intervals[in_window]
        
        if len(local_rr_intervals) > 1: # changed here => prevent misalignment
            smoothed_rmssd.append(RMSSD(local_rr_intervals)) 
        else: 
            smoothed_rmssd.append(np.nan)  # keep the time axis intact
            
    return normalized_time, smoothed_rmssd


def sliding_HR(peak_times, time_diffs, HALF_WINDOW=10):
    
    peak_times = np.array(peak_times)
    time_diffs = np.array(time_diffs)
    
    # I noticed the function skips areas close to invalid data, so I filter out NaNs values before (now it seems to work better)
    valid_data = ~np.isnan(time_diffs)
    peak_times_valid = peak_times[valid_data]
    time_diffs_valid = time_diffs[valid_data]
    
    standard_time_points = np.arange(min(peak_times_valid), max(peak_times_valid), 1)  # 1 = at each second
    
    bpm = 60 / time_diffs_valid
    
    interpolation = interp1d(peak_times_valid, bpm, kind='linear', fill_value='extrapolate') # interpolate data 
    interp_time_diffs = interpolation(standard_time_points) # at each 1s from start to end of data
    
    gaussian_smoothed_bpm = gaussian_filter1d(interp_time_diffs, sigma=HALF_WINDOW/3, mode='nearest') # apply gaussian smoothing

    return standard_time_points, gaussian_smoothed_bpm


###

# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html 


def sliding_HR_no_interp(time, rr_intervals, HALF_WINDOW=10):
    time = np.array(time)
    rr_intervals = np.array(rr_intervals)
    
    valid = ~np.isnan(rr_intervals)
    time = time[valid]
    rr_intervals = rr_intervals[valid]
    
    bpm_values = 60 / rr_intervals
    max_second = int(time[-1]) # keep only the full second of the last value in time
    normalized_time = list(range(1, max_second + 1))  # 1, 2, 3, ..., max_second
    
    smoothed_bpm = []
    
    for t in normalized_time:
        in_window = (time >= t - HALF_WINDOW) & (time <= t + HALF_WINDOW)
        local_times = time[in_window]
        local_bpm = bpm_values[in_window]

        if len(local_bpm) > 0:
            # gaussian weights, centered at t, cover ~99% distribution in the window
            weights = norm.pdf(local_times, loc=t, scale=HALF_WINDOW/3) # closer points get more weight (center at t)
            weights = weights / np.sum(weights)  # normalize so sum of weights = 1
            weighted_avg = np.sum(local_bpm * weights) # multiply each bpm to its weight (compute weighted average)
            smoothed_bpm.append(weighted_avg)
        else:
            smoothed_bpm.append(np.nan)

    return normalized_time, smoothed_bpm

        

save smoothed bpm in csv

In [6]:

# load ECG data from JSON
def load_ecg_data(file_path):
    try:
        with open(file_path, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        return {} # just in case
    
# fix time
fixed_time = np.arange(0, 150, 1)
bpm_data = {"time": fixed_time}

# target data
for target in targets:
    target_path = os.path.join(data_folder, f"{target}_RR.json")

    ecg_data_target = load_ecg_data(target_path)
    peak_times = np.array(ecg_data_target['peak_times'])
    rr_intervals = np.array(ecg_data_target['RR_intervals'])

    time_axis, smoothed_bpm = sliding_HR_no_interp(peak_times, rr_intervals, HALF_WINDOW=10)

    bpm_data[f"{target}"] = [smoothed_bpm[i] if i < len(smoothed_bpm) else np.nan for i in range(len(fixed_time))] # NaN for data outside of target's length

# perceiver data
for target in targets:
    for perceiver in perceivers:
        path = os.path.join(data_folder, f"{perceiver}_{target}_RR.json")
        if not os.path.exists(path):
            continue

        ecg_data = load_ecg_data(path)
        peak_times = np.array(ecg_data['peak_times'])
        rr_intervals = np.array(ecg_data['RR_intervals'])

        time_axis, smoothed_bpm = sliding_HR_no_interp(peak_times, rr_intervals, HALF_WINDOW=10)

        bpm_data[f"{perceiver}_{target}"] = [smoothed_bpm[i] if i < len(smoothed_bpm) else np.nan for i in range(len(fixed_time))]

# save to .csv
smoothed_bpm = pd.DataFrame(bpm_data)
smoothed_bpm.to_csv("outputs/smoothed_bpm.csv", index=False)


save rmssd through time in csv

In [7]:
# fix time
fixed_time = np.arange(0, 150, 1)
rmssd_data = {"time": fixed_time}

# targets
for target in targets:
    target_path = os.path.join(data_folder, f"{target}_RR.json")
    if not os.path.exists(target_path):
        continue

    ecg_data_target = load_ecg_data(target_path)
    peak_times = np.array(ecg_data_target['peak_times'])
    rr_intervals = np.array(ecg_data_target['RR_intervals'])

    time_axis, smoothed_rmssd = sliding_HRV(peak_times, rr_intervals)

    rmssd_data[f"{target}"] = [smoothed_rmssd[i] if i < len(smoothed_rmssd) else np.nan for i in range(len(fixed_time))] # NaN for data outside of target's length


# perceivers
for target in targets:
    for perceiver in perceivers:
        path = os.path.join(data_folder, f"{perceiver}_{target}_RR.json")
        if not os.path.exists(path):
            continue

        ecg_data = load_ecg_data(path)
        peak_times = np.array(ecg_data['peak_times'])
        rr_intervals = np.array(ecg_data['RR_intervals'])

        time_axis, smoothed_rmssd = sliding_HRV(peak_times, rr_intervals)

        rmssd_data[f"{perceiver}_{target}"] = [smoothed_rmssd[i] if i < len(smoothed_rmssd) else np.nan for i in range(len(fixed_time))]

# save to .csv
smoothed_rmssd = pd.DataFrame(rmssd_data)
smoothed_rmssd.to_csv("outputs/smoothed_rmssd.csv", index=False)


correlations between target and perceiver bpm

In [ ]:


all_output = [] # put in here scores from the perceiver-target pairs
time_shifts = np.arange(-5, 5, 0.1) 

for perceiver in perceivers:
    for target in targets:
        
        print(f"{perceiver} evaluation of {target}, checking if file exists") # tracking 
        
        perceiver_column = f"{perceiver}_{target}" # perceiver id + which target is being evaluated
        if perceiver_column not in smoothed_bpm.columns:
            continue  # perceivers did not watch all targets
                
        target_emotion = target.split('_')[1] # will use this later
        perceiver_rating = smoothed_bpm[perceiver_column].dropna()

        for any_target in targets: # do the EA scores for all targets (matching and nonmatching ones)
            any_target_emotion = any_target.split('_')[1] # will use this later
            any_target_rating = smoothed_bpm[any_target].dropna()
            print(f'any target {any_target}')
            
            # cutting to min duration between both videos (might use another method later?)
            common_data = range(min(len(perceiver_rating), len(any_target_rating))) # creates a range from 0 to min duration between both videos
            perceiver_rating_trimmed = perceiver_rating.iloc[common_data] # selects rows of data up to the shortest length
            any_target_rating_trimmed = any_target_rating.iloc[common_data] # same thing for the target

            # useful to interpolate shifted data (e.g., +100ms)
            interpol = interp1d(np.arange(len(perceiver_rating_trimmed)), perceiver_rating_trimmed, kind='linear', fill_value='extrapolate')    
                            # unsure here about fill_value: "points outside the data range will be extrapolated"
                    
            output = {
                "perceiver": perceiver, # id of the person who watched the target
                "evaluation_of": target, # target that was evaluated
                "compare_with_target": any_target, # target (specific or random) used for correlation calculation
                "emotion": target_emotion, # the emotion of the target (the right one)
                "match_type": "match" if target == any_target else "nonmatch", # match = specific target, else = nonmatch (surrogate)
                "emotion_type": "same_emotion" if target_emotion == any_target_emotion else "diff_emotion"}  # identify if the specific target and the surrogate target have the same emotion or not
                    
            r_value, p_value = spearmanr(perceiver_rating_trimmed, any_target_rating_trimmed)
            max_corr = r_value # if no better one, keep the original
            best_time_shift = 0 # if no better one, keep the original
                        
            for shift in time_shifts:
                    shifted_rating = interpol(np.arange(len(perceiver_rating_trimmed)) + shift) # interpolate perceiver rating
                    shifted_corr, shifted_p = spearmanr(shifted_rating, any_target_rating_trimmed)
                    if shifted_corr > max_corr: # if the correlation is better with the shift, save data
                        max_corr, max_p, best_time_shift = shifted_corr, shifted_p, shift
                                
            output.update({
                        "spearman_r": r_value, # original r
                        "p_value": p_value,    # original p value
                        "max_spearman_r": max_corr, # max. correlation in 10s window
                        "max_p_value": max_p, # p value associated with the max. correlation
                        "time_shift_corr": best_time_shift}) # the time shift producing the best r 
            
            all_output.append(output)
                       
                
pd.DataFrame(all_output).to_csv(f"outputs/BPM_synchrony_scores.csv", index=False)



In [ ]:
# same for RMSSD


all_output = [] # put in here scores from the perceiver-target pairs
time_shifts = np.arange(-5, 5, 0.1) 

for perceiver in perceivers:
    for target in targets:
        
        print(f"{perceiver} evaluation of {target}, checking if file exists") # tracking 
        
        perceiver_column = f"{perceiver}_{target}" # perceiver id + which target is being evaluated
        if perceiver_column not in smoothed_rmssd.columns:
            continue  # perceivers did not watch all targets
                
        target_emotion = target.split('_')[1] # will use this later
        perceiver_rating = smoothed_rmssd[perceiver_column].dropna()

        for any_target in targets: # do the EA scores for all targets (matching and nonmatching ones)
            any_target_emotion = any_target.split('_')[1] # will use this later
            any_target_rating = smoothed_rmssd[any_target].dropna()
            print(f'any target {any_target}')
            
            # cutting to min duration between both videos (might use another method later?)
            common_data = range(min(len(perceiver_rating), len(any_target_rating))) # creates a range from 0 to min duration between both videos
            perceiver_rating_trimmed = perceiver_rating.iloc[common_data] # selects rows of data up to the shortest length
            any_target_rating_trimmed = any_target_rating.iloc[common_data] # same thing for the target

            # useful to interpolate shifted data (e.g., +100ms)
            interpol = interp1d(np.arange(len(perceiver_rating_trimmed)), perceiver_rating_trimmed, kind='linear', fill_value='extrapolate')    
                            # unsure here about fill_value: "points outside the data range will be extrapolated"
                    
            output = {
                "perceiver": perceiver, # id of the person who watched the target
                "evaluation_of": target, # target that was evaluated
                "compare_with_target": any_target, # target (specific or random) used for correlation calculation
                "emotion": target_emotion, # the emotion of the target (the right one)
                "match_type": "match" if target == any_target else "nonmatch", # match = specific target, else = nonmatch (surrogate)
                "emotion_type": "same_emotion" if target_emotion == any_target_emotion else "diff_emotion"}  # identify if the specific target and the surrogate target have the same emotion or not
                    
            r_value, p_value = spearmanr(perceiver_rating_trimmed, any_target_rating_trimmed)
            max_corr = r_value # if no better one, keep the original
            best_time_shift = 0 # if no better one, keep the original
                        
            for shift in time_shifts:
                    shifted_rating = interpol(np.arange(len(perceiver_rating_trimmed)) + shift) # interpolate perceiver rating
                    shifted_corr, shifted_p = spearmanr(shifted_rating, any_target_rating_trimmed)
                    if shifted_corr > max_corr: # if the correlation is better with the shift, save data
                        max_corr, max_p, best_time_shift = shifted_corr, shifted_p, shift
                                
            output.update({
                        "spearman_r": r_value, # original r
                        "p_value": p_value,    # original p value
                        "max_spearman_r": max_corr, # max. correlation in 10s window
                        "max_p_value": max_p, # p value associated with the max. correlation
                        "time_shift_corr": best_time_shift}) # the time shift producing the best r 
            
            all_output.append(output)
                       
                
pd.DataFrame(all_output).to_csv(f"outputs/RMSSD_synchrony_scores.csv", index=False)

graphs.

In [ ]:
bpm_data = pd.read_csv("outputs/smoothed_bpm.csv")

target_emotion = lambda x : x.split("_")[1]
emotion_colour = {'anger':'red','shame':'green','neutral':'darkgray','sadness':'blue','pride':'purple','joy':'orange'}

nrow = 6 
ncol = 5 
fig, axs = plt.subplots(nrow, ncol, figsize=(13,15))
for i, ax in enumerate(fig.axes):
    
    target = targets[i]
    emotion = target_emotion(target)
    colour = emotion_colour[emotion]
    
    for perceiver in cleaned_perceivers: # plot perceivers
        if f'{perceiver}_{target}' not in bpm_data.columns:
            continue # perceivers did not watch all targets
        ax.plot(bpm_data['time'], bpm_data[f'{perceiver}_{target}'], color=colour, alpha=.2)
    ax.plot(bpm_data['time'], bpm_data[target], color=colour, label='target') 
    ax.set_xlim(0,140) # x is the same for all targets
    ax.set_ylim(40,140)
    ax.set_title(target)
 
ax.set_xlabel("t (s)")
ax.set_ylabel("heart rate (bpm)")
 
plt.tight_layout() 



##### same for rmssd

rmssd_data = pd.read_csv("outputs/smoothed_rmssd.csv")

fig, axs = plt.subplots(nrow, ncol, figsize=(13, 15))

for i, ax in enumerate(fig.axes):
    target = targets[i]
    emotion = target_emotion(target)
    colour = emotion_colour[emotion]

    for perceiver in cleaned_perceivers:
        column = f'{perceiver}_{target}'
        if column not in rmssd_data.columns:
            continue

        rmssd = rmssd_data[column]  * 1000 # to display in ms

        ax.plot(rmssd_data['time'], rmssd, color=colour, alpha=0.2)

    # target line
    target_rmssd = rmssd_data[target] * 1000
    ax.plot(rmssd_data['time'], target_rmssd, color=colour, linewidth=2)

    ax.set_xlim(0, 140)
    ax.set_ylim(0, 150) # just to make sure everything is included
    ax.set_title(target)

ax.set_xlabel("t (s)")
ax.set_ylabel("RMSSD (ms)")
plt.tight_layout()
plt.show()